# General aim
This notebook was developed as part of the Master’s thesis of June Rossen and is the second of five notebooks used to prepare data for a wind power placement multi-objective optimization.

This second notebook focuses on preparing the data of criteria used by the optimization objectives.

## Optimization constraint
The first goal of the optimization is to have an annual energy production of 100 TWh. Thus, potential AEP for each cell needs to be included first.  

## Techno-economic criteria
The criteria applied are the following:
- LCOE
- Distance to electric grid
- Distance to roads (for transport infrastructure)
- Presence of hydropower
- Presence of wind turbines
- Icing issues

## Social criteria
The criteria applied are the following:
- Viewshed study (number of people that can see a turbine, within a 55 km radius)
- Noise study (number of people within a 5 km radius)
- Smallest distance to inhabited cells
- Intersection with recreational areas
- Intersection with semi-domesticated reindeer areas

## Environmental criteria
The criteria applied are the following:
- Coverage of wetlands
- Coverage of forests
- Coverage of old forests
- Infrastructure density
- Intersection with sensitive fauna (birds)

## Notebook workflow
- First, the necessary librairies are imported, the data is loaded and irrelevant columns are removed.
- Two functions are defined to simplify the next steps (downsample and sample_centroid)
- Then, data are merged one by one to a growing geodataframe, which is then saved as a new file
    - when rasters are at a too high resolution, resample them to a 1km one, based on the shape of the SSB_land_raster file, and then sample the value at the centroid of SSB grid cells. Found it to be significantly faster than running zonal statistics (from over 1 hour to seconds)
    - additionally, if there are nodata values in the new data, the corresponding cells are removed (there is a relatively low number of nodata cells and it would impact the optimization to keep them)

In [1]:
import pandas as pd
import geopandas as gpd
import rasterio
from rasterstats import zonal_stats
from rasterio.enums import Resampling
from rasterio.mask import mask
import numpy as np
from scipy.spatial import cKDTree

In [2]:
# Load vector and csv data
NO_grid = gpd.read_file("data/SSB_tech_legal_filtered.gpkg")

In [3]:
NO_grid

,SSBid,Komm2016,Fylk2016,mean_elevation,LULC_15,LULC_16,LULC_17,LULC_18,not_covered_by_lulc,pct_NoGo_lulc,pct_slope_over_30,pct_NoGo,avg_windspeed,pop_tot,geometry
0,22960006533000,0101,01,158.045079,0.0,0.063750,0.00000,0.000000,2.081250e-01,0.271875,0.00,0.271875,7.445756,0,"MULTIPOLYGON (((297000 6533000, 296000 6533000..."
1,22970006533000,0101,01,152.303832,0.0,0.057500,0.00000,0.001250,2.050000e-01,0.263750,0.00,0.263750,7.217134,0,"MULTIPOLYGON (((298000 6533000, 297000 6533000..."
2,22980006534000,0101,01,136.235983,0.0,0.000000,0.00000,0.000000,0.000000e+00,0.000000,0.00,0.000000,6.884719,0,"MULTIPOLYGON (((299000 6534000, 298000 6534000..."
3,22990006544000,0101,01,148.310500,0.0,0.011250,0.00000,0.000000,2.220446e-16,0.011250,0.00,0.011250,7.133292,0,"MULTIPOLYGON (((300000 6544000, 299000 6544000..."
4,22990006545000,0101,01,156.253000,0.0,0.000000,0.00000,0.000000,0.000000e+00,0.000000,0.00,0.000000,7.148756,0,"MULTIPOLYGON (((300000 6545000, 299000 6545000..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
150158,31120007807000,2030,20,82.543500,0.0,0.028125,0.00000,0.000000,7.487500e-01,0.776875,0.00,0.776875,7.342870,0,"MULTIPOLYGON (((1113000 7807000, 1112000 78070..."
150159,31110007809000,2030,20,60.376000,0.0,0.030625,0.00000,0.001250,5.843750e-01,0.616250,0.01,0.626250,7.316551,0,"MULTIPOLYGON (((1112000 7809000, 1111000 78090..."
150160,31120007810000,2030,20,141.795499,0.0,0.003750,0.00000,0.000000,9.918750e-01,0.995625,0.00,0.995625,7.268112,0,"MULTIPOLYGON (((1113000 7810000, 1112000 78100..."
150161,31090007812000,2030,20,66.621500,0.0,0.038750,0.00000,0.000625,3.975000e-01,0.436875,0.14,0.576875,7.467698,0,"MULTIPOLYGON (((1110000 7812000, 1109000 78120..."


In [4]:
power_grid = gpd.read_file("data/kraftlinje_filtered.gpkg")         # will calculate distance to the closest line in the power grid
roads = gpd.read_file("data/roads_filtered.gpkg")                   # will calculate distance to the closest road in the road network

In [5]:
turbines = gpd.read_file("data/wind_turbines.gpkg")                             # assess the current presence of wind turbines in the studied cell
hydropower_plants = gpd.read_file("data/Vannkraft_Vannkraftverk.shp")           # assess the current presence of hydropower plants in the studied cell

viewshed = gpd.read_file("data/potential_turbines_visible_population_20km.gpkg")     # import the results of a viewshed (actually line-of-sight) analysis
populated_areas = gpd.read_file("data/SSB_populated.gpkg")                      # used for calculating the number of inhabitants within a 5 km radius, and for calculating the closest inhabited cell
#urban_areas = gpd.read_file("data/tettsted.gpkg")                              # in case we were interested in more compact living places and wanted to omit sparsely populated cells
recreational_areas = gpd.read_file("data/recreational_areas.gpkg")              # if intersects a cell, put 1 in the column, else 0
domesticated_reindeer = gpd.read_file("data/semidomesticated_reindeer.shp")     # if intersects a cell, put 1 in the column, else 0

lulc_stats = gpd.read_file("data/land_use_land_cover_50m_stats.csv")            # used to calculate average coverage of wetlands and forests
#lulc_stats = gpd.read_file("data/LULC_copernicus_stats.csv")                   # a different LULC dataset, less high resolution because at European level


In [6]:
# Paths to raster data
SSB_raster_path = "data/SSB_land_raster.tif"                    # used for downsampling of other rasters if the files are only at a resolution higher than 1 km
AEP_3MW_path_highres = "data/AEP_pPC_3_3_cropped_4326.tif"      # Annual Energy Production maps, for various models of wind turbines and at various resolutions
AEP_3MW_path_1km = "data/AEP_pPC_3_3_1km.tif"
AEP_5MW_path_highres = "data/AEP_pPC_5_6_cropped_4326.tif"
AEP_5MW_path_1km = "data/AEP_pPC_5_6_1km.tif"
#AEP_8MW_path_highres = "data/AEP_pPC_8_4_cropped_4326.tif"
#AEP_8MW_path_1km = "data/AEP_pPC_8_4_1km.tif"
LCOE_path = "data/lcoe_1km.tif"                                 # sampled at SSB cel centroid to estimate the costs of wind turbines
icing_path_25m = "data/ising_25m.tif"                           # 25m resolution
icing_path_1km = "data/ising_1km.tif"                           # 1km resolution, sampled at SSB cell centroid to estimate extent of icing potential

old_forest_path = "data/old_forest.tif"                                     # 16m resolution containing NaNs, used to generate the following raster without NaNs
old_forest_filled_path = "data/old_forest_noNaN.tif"                        # 16m resolution
old_forest_filled_path_1km = "data/old_forest_1km.tif"                      # 1km resolution, used to calculate average coverage of old forest over each cell
infrastructure_index_path_100m = "data/infrastructure_index_100m.tif"       # 100m resolution
infrastructure_index_path_1km = "data/infrastructure_index_1km.tif"         # 1km resolution, sampled at SSB cell centroid to estimate the "pristineness" of the nature in that cell
bird_impact_path = "data/bird_impact.tif"                                   # 1km resolution, sampled at SSB cell centroid to estimate the potential impact of wind turbines on birds in that cell

In [7]:
# To have an overview of our data
#display(NO_grid, power_grid, roads, turbines, hydropower_plants, recreational_areas, domesticated_reindeer, lulc_stats)

In [8]:
# Remove unnecessary columns
NO_grid.drop(columns=["mean_elevation", "LULC_15", "LULC_16", "LULC_17", "LULC_18", "not_covered_by_lulc", "pop_tot", "pct_NoGo_lulc", "pct_slope_over_30"], inplace=True)
NO_grid["SSBid"] = NO_grid["SSBid"].astype(int)
NO_grid["centroid"] = NO_grid.geometry.centroid

In [9]:
power_grid = power_grid[["spenningkV", "geometry"]].copy()
roads = roads[["VEGKATEGOR", "geometry"]].copy()
turbines.drop(columns=["objType", "sakTittel", "sakKategor", "objStatus", "lokalID"], inplace=True)
hydropower_plants = hydropower_plants[["status", "geometry"]].copy()

populated_areas = populated_areas[["pop_tot", "geometry"]].copy()
populated_areas["centroid"] = populated_areas.geometry.centroid
#urban_areas["centroid"] = urban_areas.geometry.centroid
domesticated_reindeer = domesticated_reindeer[["gid", "geometry"]].copy()

## Define two functions that are going to be used a few times in this notebook
- Downsampling (to the shape of the SSB 1km raster) for the rasters that have too high resolution
- Sampling of the rasters at the centroids of the SSB grid cells

In [10]:
def downsample(high_res_raster_path, low_res_raster_path, grid):
    # Open low-resolution raster (target for resampling)
    with rasterio.open(grid) as src:
        grid_meta = src.meta
        grid_transform = src.transform
        grid_crs = src.crs
        grid_shape = (src.height, src.width)

    # Open high-resolution raster
    with rasterio.open(high_res_raster_path) as src:
        data = src.read(1)

        # Reproject & resample high-res raster to match low-res raster
        resampled = np.full(grid_shape, src.nodata, dtype=np.float32)           #### CAREFUL, CHECK IF PROBLEM WITH FLOAT32 AND IF SHOULD USE FLOAT64 
        grid_meta.update(nodata=src.nodata)
        
        rasterio.warp.reproject(
            source=data,
            destination=resampled,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src.nodata,
            dst_transform=grid_transform,
            dst_crs=grid_crs,
            dst_nodata=src.nodata,
            resampling=Resampling.average  # use average of values for downsampling
        )

    # Save results to avoid having to recalculate every time
    with rasterio.open(low_res_raster_path, "w", **grid_meta) as dst:
        dst.write(resampled, 1)
    

In [11]:
def sample_centroid(raster_to_sample, vector_with_centroids, column_name):
    # create the column in the gdf
    vector_with_centroids[column_name] = np.nan
    
    with rasterio.open(raster_to_sample) as src:
        # store the nodata value
        raster_nodata = src.nodata
        print(raster_nodata)

        # check the CRS of the raster file and adjust the one of the vector file accordingly
        raster_crs = src.crs
        print(raster_crs)
        print(vector_with_centroids.crs)
        if raster_crs != vector_with_centroids.crs:
            vector_with_centroids.to_crs(raster_crs, inplace=True)
        print(vector_with_centroids.crs)

        # get the coordinates of the centroids in the vector file and sample the raster at those coordinates
        coords = [(geom.centroid.x, geom.centroid.y) for geom in vector_with_centroids.geometry]
        raster_values = [val[0] for val in src.sample(coords)]

    vector_with_centroids[column_name] = [np.nan if v == raster_nodata else v for v in raster_values]

# Techno-economic criteria

## Add the AEP map
- Start by downsampling the maps if not done yet
- Then sample the values of the three different AEP maps
- Multiply by the corresponding number of turbines that can fit in each 1km^2 cell (general estimation based on the turbine diameters and on the percentage of coverage not removed previously (due to slope and/or unfeasible LULC))
- Keep the highest of the two resulting total AEP for each cell

In [12]:
#downsample(AEP_3MW_path_highres, AEP_3MW_path_1km, SSB_raster_path)
#downsample(AEP_5MW_path_highres, AEP_5MW_path_1km, SSB_raster_path)

In [13]:
sample_centroid(AEP_3MW_path_1km, NO_grid, "avg_AEP_3MW_1turbine_kWh")
sample_centroid(AEP_5MW_path_1km, NO_grid, "avg_AEP_5MW_1turbine_kWh")

3.3999999521443642e+38
EPSG:25833
EPSG:32633
PROJCS["ETRS89 / UTM zone 33N",GEOGCS["ETRS89",DATUM["European_Terrestrial_Reference_System_1989",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6258"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4258"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",15],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","25833"]]
3.3999999521443642e+38
EPSG:25833
PROJCS["ETRS89 / UTM zone 33N",GEOGCS["ETRS89",DATUM["European_Terrestrial_Reference_System_1989",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6258"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG

In [14]:
# define the number of turbines that could be installed per cell, keeping enough spacing to avoid wake effects (gross estimation: keep 5*rotor_diameter in the wind direction and 3*rotor_diameter in the perpendicular direction) 
area_per_3MW_turbine = 0.132 * 5 * 0.132 * 3     # estimating rotor diameter of 132m, corresponds to the turbine giving the AEP map used
area_per_5MW_turbine = 0.162 * 5 * 0.162 * 3     # estimating rotor diameter of 162m, corresponds to the turbine giving the AEP map used

# one cell = 1km^2
nb_3MW_turbines_per_cell = 1 / area_per_3MW_turbine
nb_5MW_turbines_per_cell = 1 / area_per_5MW_turbine

print(nb_3MW_turbines_per_cell, nb_5MW_turbines_per_cell)

# factor in the percentage of each cell that is unfeasible (because of slope and/or LULC) and multiply that by the number of turbines that could be put, keeping enough spacing 
NO_grid["potential_nb_3MW_turbines"] = round((1 - NO_grid["pct_NoGo"]) * nb_3MW_turbines_per_cell)
NO_grid["potential_nb_5MW_turbines"] = round((1 - NO_grid["pct_NoGo"]) * nb_5MW_turbines_per_cell)

# multiply the max number of turbines that can be placed by the average AEP. Convert from kWh to GWh for readability
NO_grid["avg_AEP_3MW"] = NO_grid["potential_nb_3MW_turbines"] * NO_grid["avg_AEP_3MW_1turbine_kWh"] / 1_000_000
NO_grid["avg_AEP_5MW"] = NO_grid["potential_nb_5MW_turbines"] * NO_grid["avg_AEP_5MW_1turbine_kWh"] / 1_000_000

NO_grid = NO_grid.dropna(subset=["avg_AEP_3MW", "avg_AEP_5MW"]).reset_index(drop=True)

NO_grid["Best_turbine_model"] = NO_grid[["avg_AEP_3MW", "avg_AEP_5MW"]].idxmax(axis=1)
NO_grid["nb_turbines"] = np.where(
    NO_grid["Best_turbine_model"] == "avg_AEP_3MW",
    NO_grid["potential_nb_3MW_turbines"],
    NO_grid["potential_nb_5MW_turbines"]
)
NO_grid["AEP_GWh"] = NO_grid[["avg_AEP_3MW", "avg_AEP_5MW"]].max(axis=1)

# to analyse differences in turbine choice
NO_grid["pct_difference_turbine_choice_3-5"] = (NO_grid["avg_AEP_3MW"] - NO_grid["avg_AEP_5MW"]) / NO_grid["avg_AEP_3MW"]

3.826140189776553 2.540263171264543


In [16]:
# Whithout the 8 MW model:                                      total of 6.4e6 GWh = 6.4e3 TWh = 6'400 Twh = 64x final goal!
NO_grid["AEP_GWh"].agg(["sum", "mean", "min", "max", "std"])
# With 8MW model:                                               total of 8.0e6 GWh = 8.0e3 TWh = 8'000 Twh = 80x final goal!

sum     5.695872e+06
mean    3.794137e+01
min     0.000000e+00
max     1.007428e+02
std     2.041981e+01
Name: AEP_GWh, dtype: float64

In [17]:
NO_grid["avg_AEP_3MW"].agg(["sum", "mean", "min", "max", "std"])        # total of 5.1e6 GWh = 5.1e3 TWh = 5'100 Twh = 51x final goal!

sum     4.928981e+06
mean    3.283295e+01
min     0.000000e+00
max     8.103463e+01
std     1.789005e+01
Name: avg_AEP_3MW, dtype: float64

In [18]:
NO_grid["avg_AEP_5MW"].agg(["sum", "mean", "min", "max", "std"])        # total of 6.4e6 GWh = 6.4e3 TWh = 6'400 Twh = 64x final goal!

sum     5.378192e+06
mean    3.582523e+01
min     0.000000e+00
max     1.007428e+02
std     2.064791e+01
Name: avg_AEP_5MW, dtype: float64

In [ ]:
# in case we want to analyse the results of AEP
#NO_grid_temp = NO_grid.drop(columns="centroid")
#NO_grid_temp.to_file("data/SSB_tech_legal_filtered_AEP_for_analysis.gpkg", driver="GPKG")

In [21]:
NO_grid = NO_grid[NO_grid["AEP_GWh"] > 0].reset_index(drop=True)      # reduces from 150'123 to 139'677 grid cells

In [22]:
NO_grid

,SSBid,Komm2016,Fylk2016,pct_NoGo,avg_windspeed,geometry,centroid,avg_AEP_3MW_1turbine_kWh,avg_AEP_5MW_1turbine_kWh,potential_nb_3MW_turbines,potential_nb_5MW_turbines,avg_AEP_3MW,avg_AEP_5MW,Best_turbine_model,nb_turbines,AEP_GWh,pct_difference_turbine_choice_3-5
0,22960006533000,0101,01,0.271875,7.445756,"MULTIPOLYGON (((297000 6533000, 296000 6533000...",POINT (296500 6533500),11572505.0,19234886.0,3.0,2.0,34.717515,38.469772,avg_AEP_5MW,2.0,38.469772,-0.108080
1,22970006533000,0101,01,0.263750,7.217134,"MULTIPOLYGON (((298000 6533000, 297000 6533000...",POINT (297500 6533500),11365627.0,18886952.0,3.0,2.0,34.096881,37.773904,avg_AEP_5MW,2.0,37.773904,-0.107840
2,22980006534000,0101,01,0.000000,6.884719,"MULTIPOLYGON (((299000 6534000, 298000 6534000...",POINT (298500 6534500),11046537.0,18347876.0,4.0,3.0,44.186148,55.043628,avg_AEP_5MW,3.0,55.043628,-0.245721
3,22990006544000,0101,01,0.011250,7.133292,"MULTIPOLYGON (((300000 6544000, 299000 6544000...",POINT (299500 6544500),11259643.0,18712946.0,4.0,3.0,45.038572,56.138838,avg_AEP_5MW,3.0,56.138838,-0.246461
4,22990006545000,0101,01,0.000000,7.148756,"MULTIPOLYGON (((300000 6545000, 299000 6545000...",POINT (299500 6545500),11783823.0,19610206.0,4.0,3.0,47.135292,58.830618,avg_AEP_5MW,3.0,58.830618,-0.248122
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139117,31140007801000,2030,20,0.838125,7.405538,"MULTIPOLYGON (((1115000 7801000, 1114000 78010...",POINT (1114500 7801500),10539202.0,17547292.0,1.0,0.0,10.539202,0.000000,avg_AEP_3MW,1.0,10.539202,1.000000
139118,31140007802000,2030,20,0.823750,7.383948,"MULTIPOLYGON (((1115000 7802000, 1114000 78020...",POINT (1114500 7802500),10560500.0,17589724.0,1.0,0.0,10.560500,0.000000,avg_AEP_3MW,1.0,10.560500,1.000000
139119,31120007807000,2030,20,0.776875,7.342870,"MULTIPOLYGON (((1113000 7807000, 1112000 78070...",POINT (1112500 7807500),9523185.0,15830215.0,1.0,1.0,9.523185,15.830215,avg_AEP_5MW,1.0,15.830215,-0.662282
139120,31110007809000,2030,20,0.626250,7.316551,"MULTIPOLYGON (((1112000 7809000, 1111000 78090...",POINT (1111500 7809500),8722024.0,14465424.0,1.0,1.0,8.722024,14.465424,avg_AEP_5MW,1.0,14.465424,-0.658494


## Add the LCOE to the gdf

In [23]:
sample_centroid(LCOE_path, NO_grid, "avg_LCOE")

NO_grid = NO_grid[~np.isnan(NO_grid["avg_LCOE"])].reset_index(drop=True).copy()   # reduces from 139'677 to 136'185 grid cells

-3.4028230607370965e+38
EPSG:32633
PROJCS["ETRS89 / UTM zone 33N",GEOGCS["ETRS89",DATUM["European_Terrestrial_Reference_System_1989",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6258"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4258"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",15],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","25833"]]
PROJCS["WGS 84 / UTM zone 33N",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER

## Add distance to the electric grid
Removed the power lines that had "spenningkV" of NULL or 0 (around 800 lines out of 146'000), assuming it meant the line were not constructed yet.

In [24]:
NO_grid = NO_grid.set_geometry("centroid")          # use the centroid of the cells as the comparison point

if power_grid.crs != NO_grid.crs:
    power_grid = power_grid.to_crs(NO_grid.crs)

nearest = gpd.sjoin_nearest(
    NO_grid,
    power_grid,
    how="left",
    distance_col="dist_to_grid"
)
NO_grid = nearest.drop(columns="index_right").copy()
NO_grid = NO_grid.set_geometry("geometry")
NO_grid = NO_grid.drop_duplicates(subset="SSBid") # sjoin_nearest duplicates lines when there are several power lines / roads that are equistant to a cell's centroid. Drop the duplicates based on SSBid


## Add distance to roads
Legend of road types:
+ E europaveg
+ rv. riksveg
+ fv. fylkesveg
+ kv. kommunal veg
+ pv. privat veg
+ sv. skogsbilveg
+ arm sideveg 

--> kept E, rv., fv. and kv. in a previous filter made in QGIS, assuming those roads are large and strong enough to handle wind turbine transport

In [25]:
NO_grid = NO_grid.set_geometry("centroid")       # use the centroid of the cells as the comparison point

if roads.crs != NO_grid.crs:
    roads = roads.to_crs(NO_grid.crs)

nearest = gpd.sjoin_nearest(
    NO_grid,
    roads,
    how="left",
    distance_col="dist_to_road"
)

NO_grid = nearest.drop(columns="index_right").copy()
NO_grid = NO_grid.set_geometry("geometry")
NO_grid = NO_grid.drop_duplicates(subset="SSBid")

## Add the number of existing wind turbines over each cell


In [26]:
NO_grid["nb_existing_turbines"] = np.nan

if NO_grid.crs != turbines.crs:
    turbines = turbines.to_crs(NO_grid.crs)

turbines_in_grid = gpd.sjoin(turbines, NO_grid, how="left", predicate="within")
counts = turbines_in_grid.groupby("index_right").size()
NO_grid["nb_existing_turbines"] = NO_grid.index.map(counts).fillna(0).astype(int)

## Add the number of existing hydropower plants over each cell
Need to remove some of the plants from the initial file. Legend of the status:
+ Drift = D
+ Nedlagt = N
+ Ombygd = O
+ Planlagt = P
+ Planlagt illustrert = P1
+ Planlagt, prosjektert = P2
+ Under arbeid = U
+ Vedtatt = V
+ Fjernet = FJ

--> Keep D, O, U

In [27]:
NO_grid["nb_existing_hydro_plants"] = np.nan

status_of_interest = ["D", "O", "U"]

hydropower_plants_filtered = hydropower_plants[hydropower_plants["status"].isin(status_of_interest)]

if NO_grid.crs != hydropower_plants_filtered.crs:
    hydropower_plants_filtered = hydropower_plants_filtered.to_crs(NO_grid.crs)

hydro_in_grid = gpd.sjoin(hydropower_plants_filtered, NO_grid, how="left", predicate="within")
counts = hydro_in_grid.groupby("index_right").size()
NO_grid["nb_existing_hydro_plants"] = NO_grid.index.map(counts).fillna(0).astype(int)

## Add the average value of icing (in hours/year) per grid cell

If 1km resolution raster is not provided, run the first cell.

If zonal statistics is preferred to sampling the centroid of SSB grid cells, only run the third cell.

In [26]:
#downsample(icing_path_25m, icing_path_1km, SSB_raster_path)

In [28]:
sample_centroid(icing_path_1km, NO_grid, "icing_potential")
NO_grid = NO_grid[~np.isnan(NO_grid["icing_potential"])].reset_index(drop=True).copy()   # reduces from 136'185 to 136'083 grid cells

-1.0
EPSG:25833
PROJCS["WGS 84 / UTM zone 33N",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",15],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","32633"]]
PROJCS["ETRS89 / UTM zone 33N",GEOGCS["ETRS89",DATUM["European_Terrestrial_Reference_System_1989",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6258"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4258"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origi

In [28]:
### compared the methods: less than 10% of the cells have more than 1% difference between the two methods (cf. file "sensitivity_zonalstats_vs_downsampling.gpkg" if curious)
"""
# Calculate the mean coverage of icing pre grid cell
stats = zonal_stats(
    vectors = NO_grid,
    raster = icing_path_25m,
    stats = "mean",
    all_touched=False,      # only count cells whose center is inside polygon
    geojson_out=False,      # don’t return full geometries
)

NO_grid["icing_potential_zonal"] = [s["mean"] for s in stats]

# If one wants to compare the two methods
NO_grid["icing_differences_method"] = (NO_grid["icing_potential"] - NO_grid["icing_potential_zonal"]) / NO_grid["icing_potential"]
NO_grid_temp = NO_grid[["SSBid", "icing_potential", "icing_potential_zonal", "icing_differences_method", "geometry"]]
NO_grid_temp.to_file("sensitivity_zonalstats_vs_downsampling_icing.gpkg", driver="GPKG")

"""

'\n# Calculate the mean coverage of icing pre grid cell\nstats = zonal_stats(\n    vectors = NO_grid,\n    raster = icing_path_25m,\n    stats = "mean",\n    all_touched=False,      # only count cells whose center is inside polygon\n    geojson_out=False,      # don’t return full geometries\n)\n\nNO_grid["icing_potential_zonal"] = [s["mean"] for s in stats]\n\n# If one wants to compare the two methods\nNO_grid["icing_differences_method"] = (NO_grid["icing_potential"] - NO_grid["icing_potential_zonal"]) / NO_grid["icing_potential"]\nNO_grid_temp = NO_grid[["SSBid", "icing_potential", "icing_potential_zonal", "icing_differences_method", "geometry"]]\nNO_grid_temp.to_file("sensitivity_zonalstats_vs_downsampling_icing.gpkg", driver="GPKG")\n\n'

# Social criteria

## Add viewshed analysis results

In [29]:
NO_grid = NO_grid.merge(
    viewshed[["SSBid", "visible_pop"]],
    left_on="SSBid",        # Column in the gdf
    right_on="SSBid",       # Column in the df
    how="left"              # Use 'left' to keep all rows from the gdf
)

## Add number of people within a 5 km radius

In [30]:
d_max = 5000   # set 5 km as the distance within which noise could be a problem

if populated_areas.crs != NO_grid.crs:
    populated_areas = populated_areas.to_crs(NO_grid.crs)

turb_coords = np.array([[geom.centroid.x, geom.centroid.y] for geom in NO_grid.geometry])       # use the centroid of the cells as the comparison point
pop_coords = np.array([[geom.centroid.x, geom.centroid.y] for geom in populated_areas.geometry])

# KD-tree allows for a faster look-up using query_ball_point
turb_tree = cKDTree(turb_coords)

# For each population cell, find turbines within d_max
neighbor_5km_lists = turb_tree.query_ball_point(pop_coords, r=d_max)

pairs_5km = [(turb_i, pop_i)
         for pop_i, turb_list in enumerate(neighbor_5km_lists)
         for turb_i in turb_list]

print(f"Candidate pairs after horizon filter: {len(pairs_5km):,}")

# make DataFrame from visible pairs
df_5km = pd.DataFrame(pairs_5km, columns=["turb_idx", "pop_idx"])

# map population counts
df_5km["pop_tot"] = populated_areas.loc[df_5km["pop_idx"], "pop_tot"].values

# sum inhabitants per turbine
counts_5km = df_5km.groupby("turb_idx")["pop_tot"].sum()

# add column to turbine gdf
NO_grid["population_in_5km"] = NO_grid.index.map(counts_5km).fillna(0).astype(int)

Candidate pairs after horizon filter: 892,527


## Add smallest distance to inhabited cells

In [31]:
NO_grid = NO_grid.set_geometry("centroid")
populated_areas_centroid = populated_areas.set_geometry("centroid")
populated_areas_centroid.drop(columns="geometry", inplace=True)

if populated_areas_centroid.crs != NO_grid.crs:
    NO_grid = NO_grid.to_crs(populated_areas_centroid.crs)

nearest = gpd.sjoin_nearest(
    NO_grid,
    populated_areas_centroid,
    how="left",
    distance_col="dist_to_individuals"
)

NO_grid = nearest.drop(columns=["index_right", "pop_tot"]).copy()
NO_grid = NO_grid.set_geometry("geometry")
NO_grid = NO_grid.drop_duplicates(subset="SSBid")


Add distance to urban areas -> not relevant anymore, good with distance to populated areas

In [32]:
# NO_grid = NO_grid.set_geometry("centroid")
# #urban_areas_centroid = urban_areas.set_geometry("centroid")
# #urban_areas_centroid.drop(columns="geometry", inplace=True)

# #if urban_areas_centroid.crs != NO_grid.crs:
# #    NO_grid = NO_grid.to_crs(urban_areas_centroid.crs)

# if urban_areas.crs != NO_grid.crs:
#     urban_areas = urban_areas.to_crs(NO_grid.crs)

# nearest = gpd.sjoin_nearest(
#     NO_grid,
#     urban_areas,
#     how="left",
#     distance_col="dist_to_urban"
# )

# NO_grid = nearest.drop(columns="index_right").copy()
# NO_grid = NO_grid.set_geometry("geometry")
# NO_grid = NO_grid.drop_duplicates(subset="SSBid")


## Add if each cell intersects recreational areas

In [32]:
NO_grid["recreational_area"] = 0

if recreational_areas.crs != NO_grid.crs:
    recreational_areas = recreational_areas.to_crs(NO_grid.crs)

# Spatial join: find grid cells that intersect or touch recreational areas
joined = gpd.sjoin(NO_grid, recreational_areas, how="inner", predicate="intersects")

# Identify IDs of grid cells that touch/intersect recreational areas
bad_ids = joined.index.unique()

# Mark those cells
NO_grid.loc[bad_ids, "recreational_area"] = 1

## Add if each cell intersects semi-domesticated reindeer areas

In [33]:
NO_grid["domesticated_reindeer"] = 0

if domesticated_reindeer.crs != NO_grid.crs:
    domesticated_reindeer = domesticated_reindeer.to_crs(NO_grid.crs)

# Spatial join: find grid cells that intersect or touch recreational areas
joined = gpd.sjoin(NO_grid, domesticated_reindeer, how="inner", predicate="intersects")

# Identify IDs of grid cells that touch/intersect recreational areas
bad_ids = joined.index.unique()

# Mark those cells
NO_grid.loc[bad_ids, "domesticated_reindeer"] = 1

# Environmental criteria

## Add the coverage of wetlands and forests LULC

Sample the percentage coverage of the corresponding LULC classes (previously calculated) and add the different wetland types and forest types together.

In [34]:
# With the AR50 LULC

NO_grid["wetlands"] = np.nan
NO_grid["forest"] = np.nan

# Need to convert values from type object to type float
lulc_stats = lulc_stats.astype(float)
lulc_stats["SSBid"] = lulc_stats["SSBid"].astype(int)

# Merge by SSBid to avoid any potential problems with indices
NO_grid = NO_grid.merge(
    lulc_stats[["SSBid", "LULC_3", "LULC_4", "LULC_5", "LULC_6", "LULC_7", "LULC_11", "LULC_12"]],
    on="SSBid",
    how="left"
)

NO_grid["wetlands"] = NO_grid["LULC_11"] + NO_grid["LULC_12"]
NO_grid["forest"] = NO_grid["LULC_3"] + NO_grid["LULC_4"] + NO_grid["LULC_5"] + NO_grid["LULC_6"] + NO_grid["LULC_7"]

NO_grid.drop(columns=["LULC_3", "LULC_4", "LULC_5", "LULC_6", "LULC_7", "LULC_11", "LULC_12"], inplace=True)


In [36]:
# With the Copernicus LULC
"""
NO_grid["wetlands"] = np.nan
NO_grid["forest"] = np.nan

# Need to convert values from type object to type float
lulc_stats = lulc_stats.astype(float)
lulc_stats["SSBid"] = lulc_stats["SSBid"].astype(int)

# Merge by SSBid to avoid any potential problems with indices
NO_grid = NO_grid.merge(
    lulc_stats[["SSBid", "LULC_9", "LULC_11", "LULC_12"]],
    on="SSBid",
    how="left"
)

NO_grid["wetlands"] = NO_grid["LULC_9"]
NO_grid["forest"] = NO_grid["LULC_11"] + NO_grid["LULC_12"]

NO_grid.drop(columns=["LULC_9", "LULC_11", "LULC_12"], inplace=True)
"""

'\nNO_grid["wetlands"] = np.nan\nNO_grid["forest"] = np.nan\n\n# Need to convert values from type object to type float\nlulc_stats = lulc_stats.astype(float)\nlulc_stats["SSBid"] = lulc_stats["SSBid"].astype(int)\n\n# Merge by SSBid to avoid any potential problems with indices\nNO_grid = NO_grid.merge(\n    lulc_stats[["SSBid", "LULC_9", "LULC_11", "LULC_12"]],\n    on="SSBid",\n    how="left"\n)\n\nNO_grid["wetlands"] = NO_grid["LULC_9"]\nNO_grid["forest"] = NO_grid["LULC_11"] + NO_grid["LULC_12"]\n\nNO_grid.drop(columns=["LULC_9", "LULC_11", "LULC_12"], inplace=True)\n'

## Add the coverage of old forests

Careful to take a file where the nodata is replaced by 0! Instead of using the first cell, this was done with QGIS, as doing it with Python took too much memory for my computer.

If 1km resolution raster is not provided, run the second cell.

If zonal statistics is preferred to sampling the centroid of SSB grid cells, run the fourth cell.

In [37]:
"""
# cell to replace nodata values by 0, if computer's memory allows it
with rasterio.open(old_forest_path) as src:
    data = src.read(1, masked=True)
    
    # replace nodata (masked) with 0
    filled = np.where(data.mask, 0, data)
    
    profile = src.profile
    profile.update(nodata=None)  # clear nodata flag so 0 won't be ignored

    # save the raster with nodata filled as 0
    old_forest_filled_path = "data/old_forest_noNaN.tif"

with rasterio.open(old_forest_filled_path, "w", **profile) as dst:
    dst.write(filled, 1)
"""

'\n# cell to replace nodata values by 0, if computer\'s memory allows it\nwith rasterio.open(old_forest_path) as src:\n    data = src.read(1, masked=True)\n\n    # replace nodata (masked) with 0\n    filled = np.where(data.mask, 0, data)\n\n    profile = src.profile\n    profile.update(nodata=None)  # clear nodata flag so 0 won\'t be ignored\n\n    # save the raster with nodata filled as 0\n    old_forest_filled_path = "data/old_forest_noNaN.tif"\n\nwith rasterio.open(old_forest_filled_path, "w", **profile) as dst:\n    dst.write(filled, 1)\n'

In [38]:
#downsample(old_forest_filled_path, old_forest_filled_path_1km, SSB_raster_path)

In [35]:
sample_centroid(old_forest_filled_path_1km, NO_grid, "old_forest")

None
EPSG:25833
PROJCS["ETRS89 / UTM zone 33N",GEOGCS["ETRS89",DATUM["European_Terrestrial_Reference_System_1989",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6258"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4258"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",15],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","25833"]]
PROJCS["ETRS89 / UTM zone 33N",GEOGCS["ETRS89",DATUM["European_Terrestrial_Reference_System_1989",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6258"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4258"]],PROJECTION["Transverse_Mer

In [40]:
### compared the methods: few cells have more than 1% difference between the two methods (cf. file "sensitivity_zonalstats_vs_downsampling_oldforest.gpkg" if curious),
# and those that have often have very low old forest coverage, so the difference is probably due to rounding/border/computation effects

# Calculate the mean coverage of old forest per grid cell
"""
stats = zonal_stats(
    vectors = NO_grid,
    raster = old_forest_filled_path,
    stats = "mean",
    all_touched=False,      # only count cells whose center is inside polygon
    geojson_out=False,      # don’t return full geometries
)

NO_grid["old_forest_zonal"] = [s["mean"] for s in stats]

NO_grid["old_forest_differences_method"] = (NO_grid["old_forest"] - NO_grid["old_forest_zonal"]) / NO_grid["old_forest"]
NO_grid_temp = NO_grid[["SSBid", "old_forest", "old_forest_zonal", "old_forest_differences_method", "geometry"]]
NO_grid_temp.to_file("sensitivity_zonalstats_vs_downsampling_oldforest.gpkg", driver="GPKG")
"""

'\nstats = zonal_stats(\n    vectors = NO_grid,\n    raster = old_forest_filled_path,\n    stats = "mean",\n    all_touched=False,      # only count cells whose center is inside polygon\n    geojson_out=False,      # don’t return full geometries\n)\n\nNO_grid["old_forest_zonal"] = [s["mean"] for s in stats]\n\nNO_grid["old_forest_differences_method"] = (NO_grid["old_forest"] - NO_grid["old_forest_zonal"]) / NO_grid["old_forest"]\nNO_grid_temp = NO_grid[["SSBid", "old_forest", "old_forest_zonal", "old_forest_differences_method", "geometry"]]\nNO_grid_temp.to_file("sensitivity_zonalstats_vs_downsampling_oldforest.gpkg", driver="GPKG")\n'

## Add the infrastructure index

Use a downsampled (1km) infrastructure index layer and sample the centroid of the SSB cells.

In [41]:
#downsample(infrastructure_index_path_100m, infrastructure_index_path_1km, SSB_raster_path)

In [36]:
sample_centroid(infrastructure_index_path_1km, NO_grid, "infrastructure_index")
NO_grid = NO_grid[~np.isnan(NO_grid["infrastructure_index"])].reset_index(drop=True).copy()   # no reduction in cell number

-3.4028230607370965e+38
EPSG:25833
PROJCS["ETRS89 / UTM zone 33N",GEOGCS["ETRS89",DATUM["European_Terrestrial_Reference_System_1989",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6258"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4258"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",15],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","25833"]]
PROJCS["ETRS89 / UTM zone 33N",GEOGCS["ETRS89",DATUM["European_Terrestrial_Reference_System_1989",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6258"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4258"]],PROJECT

## Add the bird impact potential
Use a downsampled (1km) bird impact potential layer and sample the centroid of the SSB cells.

In [37]:
sample_centroid(bird_impact_path, NO_grid, "bird_impact_potential")
NO_grid = NO_grid[~np.isnan(NO_grid["bird_impact_potential"])].reset_index(drop=True).copy()   # reduces from 136'083 to 134'660 grid cells

None
EPSG:3035
PROJCS["ETRS89 / UTM zone 33N",GEOGCS["ETRS89",DATUM["European_Terrestrial_Reference_System_1989",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6258"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4258"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",15],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","25833"]]
PROJCS["ETRS89-extended / LAEA Europe",GEOGCS["ETRS89",DATUM["European_Terrestrial_Reference_System_1989",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6258"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4258"]],PROJECTION["Lambert

In [38]:
NO_grid

,SSBid,Komm2016,Fylk2016,pct_NoGo,avg_windspeed,geometry,centroid,avg_AEP_3MW_1turbine_kWh,avg_AEP_5MW_1turbine_kWh,potential_nb_3MW_turbines,...,visible_pop,population_in_5km,dist_to_individuals,recreational_area,domesticated_reindeer,wetlands,forest,old_forest,infrastructure_index,bird_impact_potential
0,22990006544000,0101,01,1.125000e-02,7.133292,"MULTIPOLYGON (((4408499.644 3988462.997, 44075...",POINT (299500 6544500),11259643.0,18712946.0,4.0,...,29555,458,1000.000000,1,0,0.000000,0.988750,0.164096,2.672269,0.222749
1,22990006545000,0101,01,0.000000e+00,7.148756,"MULTIPOLYGON (((4408425.191 3989458.236, 44074...",POINT (299500 6545500),11783823.0,19610206.0,4.0,...,30227,472,1000.000000,1,0,0.000000,0.999375,0.200320,2.571814,0.219633
2,22990006550000,0101,01,1.937500e-02,7.336300,"MULTIPOLYGON (((4408052.815 3994434.348, 44070...",POINT (299500 6550500),11115102.0,18448636.0,4.0,...,31333,776,1000.000000,0,0,0.297500,0.682500,0.041600,3.707362,0.186828
3,22980006555000,0101,01,1.110223e-16,7.359521,"MULTIPOLYGON (((4406681.094 3999336.954, 44056...",POINT (298500 6555500),11519288.0,19160622.0,4.0,...,32864,4709,1000.000000,1,0,0.000000,0.966875,0.316928,3.491421,0.171705
4,22990006555000,0101,01,1.110223e-16,7.435547,"MULTIPOLYGON (((4407680.258 3999410.323, 44066...",POINT (299500 6555500),12323313.0,20524556.0,4.0,...,32504,1609,2000.000000,1,0,0.015000,0.985000,0.357888,3.337394,0.170284
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134278,31040007818000,2030,20,5.168750e-01,9.022841,"MULTIPOLYGON (((5120420.194 5303057.846, 51194...",POINT (1104500 7818500),17134344.0,28635602.0,2.0,...,0,0,13416.407865,0,1,0.000000,0.190000,0.006784,0.520901,0.137528
134279,31050007818000,2030,20,4.856250e-01,8.734272,"MULTIPOLYGON (((5121425.654 5303108.24, 512042...",POINT (1105500 7818500),18009040.0,29959542.0,2.0,...,0,0,14317.821063,1,1,0.000000,0.140625,0.000000,1.777281,0.105249
134280,31020007819000,2030,20,3.825000e-01,8.457061,"MULTIPOLYGON (((5118336.321 5303939.79, 511733...",POINT (1102500 7819500),16562781.0,27648840.0,2.0,...,0,0,12206.555616,0,1,0.000000,0.150000,0.017280,0.221662,0.273842
134281,31030007819000,2030,20,1.900000e-01,8.727259,"MULTIPOLYGON (((5119341.831 5303990.254, 51183...",POINT (1103500 7819500),18427086.0,30659742.0,3.0,...,0,0,13038.404811,1,1,0.038125,0.016250,0.000000,1.156375,0.149288


In [39]:
NO_grid.drop(columns="centroid", inplace=True)
NO_grid.to_file("data/SSB_techlegalfiltered_opticriteria.gpkg", driver="GPKG")